In [16]:
import sys
from pathlib import Path

current = Path.cwd().resolve()

if current.name == "notebooks":
    PROJECT_ROOT = current.parent
elif (current / "src").exists():
    PROJECT_ROOT = current
else:
    PROJECT_ROOT = current

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config.paths import *

In [ ]:
# ============================================================
# PHASE 0 — PATH CONFIGURATION
# ============================================================

from pathlib import Path
import os

PROJECT_ROOT = Path(
    os.getenv(
        "PROJECT_ROOT",
        PROJECT_ROOT 
    )
)

DATA_DIR = PROJECT_ROOT / "data"
DATA_RAW_DIR = DATA_DIR / "raw"
DATA_CLEAN_DIR = DATA_DIR / "clean"

DOCS_DIR = PROJECT_ROOT / "docs"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

SRC_DIR = PROJECT_ROOT / "src"

# Keep old variable name to avoid breaking notebook code
MODEL_DIR = PROJECT_ROOT / "models"

REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"

CLASSICAL_FIGURES_DIR = FIGURES_DIR / "classical"
CLASSICAL_TABLES_DIR = TABLES_DIR / "classical"

# Keep old variable names to avoid breaking notebook code
CLASSICAL_REPORTS_DIR = CLASSICAL_TABLES_DIR
TRANSFORMER_REPORTS_DIR = TABLES_DIR / "transformers"
COMPARISON_REPORTS_DIR = CLASSICAL_TABLES_DIR

RAW_DATA_PATH = DATA_RAW_DIR / "Mental Health Disorder Detection Dataset.csv"
CLEAN_DATA_PATH = DATA_CLEAN_DIR / "mental_health_detection_clean.csv"

NORMAL_CV_RESULTS_PATH = CLASSICAL_REPORTS_DIR / "normal_cv_summary.csv"
NESTED_CV_RESULTS_PATH = CLASSICAL_REPORTS_DIR / "nested_cv_summary.csv"
COMPARISON_RESULTS_PATH = COMPARISON_REPORTS_DIR / "classical_model_comparison.csv"
FINAL_TEST_RESULTS_PATH = CLASSICAL_REPORTS_DIR / "final_test_metrics.csv"
CLASS_RECALL_RESULTS_PATH = CLASSICAL_REPORTS_DIR / "class_recall_results.csv"
FINAL_PREDICTIONS_PATH = CLASSICAL_REPORTS_DIR / "final_test_predictions.csv"
FINAL_MODEL_PATH = MODEL_DIR / "best_classical_model.joblib"

for directory in [
    DATA_RAW_DIR,
    DATA_CLEAN_DIR,
    DOCS_DIR,
    NOTEBOOKS_DIR,
    SRC_DIR,
    MODEL_DIR,
    CLASSICAL_REPORTS_DIR,
    TRANSFORMER_REPORTS_DIR,
    COMPARISON_REPORTS_DIR,
    CLASSICAL_FIGURES_DIR,
    CLASSICAL_TABLES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("✅ Project paths ready")
print(f"PROJECT_ROOT            : {PROJECT_ROOT}")
print(f"RAW_DATA_PATH           : {RAW_DATA_PATH}")
print(f"CLEAN_DATA_PATH         : {CLEAN_DATA_PATH}")
print(f"NORMAL_CV_RESULTS_PATH  : {NORMAL_CV_RESULTS_PATH}")
print(f"NESTED_CV_RESULTS_PATH  : {NESTED_CV_RESULTS_PATH}")
print(f"COMPARISON_RESULTS_PATH : {COMPARISON_RESULTS_PATH}")
print(f"FINAL_TEST_RESULTS_PATH : {FINAL_TEST_RESULTS_PATH}")
print(f"CLASS_RECALL_RESULTS    : {CLASS_RECALL_RESULTS_PATH}")
print(f"FINAL_PREDICTIONS_PATH  : {FINAL_PREDICTIONS_PATH}")
print(f"FINAL_MODEL_PATH        : {FINAL_MODEL_PATH}")

✅ Project paths ready
PROJECT_ROOT            : C:\Users\anafi\Desktop\final_project_jedha
RAW_DATA_PATH           : C:\Users\anafi\Desktop\final_project_jedha\data\raw\Mental Health Disorder Detection Dataset.csv
CLEAN_DATA_PATH         : C:\Users\anafi\Desktop\final_project_jedha\data\clean\mental_health_detection_clean.csv
NORMAL_CV_RESULTS_PATH  : C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\normal_cv_summary.csv
NESTED_CV_RESULTS_PATH  : C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\nested_cv_summary.csv
COMPARISON_RESULTS_PATH : C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\classical_model_comparison.csv
FINAL_TEST_RESULTS_PATH : C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\final_test_metrics.csv
CLASS_RECALL_RESULTS    : C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\class_recall_results.csv
FINAL_PREDICTIONS_PATH  : C:\Users\anafi\Desktop\final_project_jedha\reports\ta

# Benchmark Classique — Modèles TF-IDF + ML

## Objectif
Ce notebook évalue plusieurs modèles classiques de classification de texte sur deux variantes du corpus :
- `raw`
- `masked`

## Objectifs méthodologiques
- comparer plusieurs baselines robustes,
- intégrer une logique de priorité clinique via `critical_recall`,
- comparer validation simple et nested CV,
- sélectionner un modèle champion,
- produire des exports propres pour l’évaluation clinique finale.

## Remarque
Les noms de variables et fonctions du pipeline sont conservés pour assurer la compatibilité avec les notebooks suivants.

In [18]:
# ============================================================
# PHASE 0 — IMPORTS & GLOBAL SETTINGS
# ============================================================

import json
import joblib
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from IPython.display import display

from sklearn.base import clone
from sklearn.metrics import classification_report, f1_score, recall_score, confusion_matrix
from sklearn.model_selection import ParameterGrid, StratifiedKFold, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TEXT_COL = "body"
MASKED_COL = "body_masked"
TARGET_COL = "category"

PLOTLY_TEMPLATE = "plotly_white"

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

print("✅ Imports loaded")
print(f"TEXT_COL   = {TEXT_COL}")
print(f"MASKED_COL = {MASKED_COL}")
print(f"TARGET_COL = {TARGET_COL}")

✅ Imports loaded
TEXT_COL   = body
MASKED_COL = body_masked
TARGET_COL = category


## Chargement du dataset clean

Cette section charge le dataset final préparé dans le notebook 1 et vérifie la présence des colonnes nécessaires au benchmark classique.

In [19]:
# ============================================================
# PHASE 1 — LOAD CLEAN DATASET
# ============================================================

df = pd.read_csv(CLEAN_DATA_PATH)

required_cols = [TEXT_COL, MASKED_COL, TARGET_COL]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in clean dataset: {missing_cols}")

for col in required_cols:
    df[col] = df[col].apply(lambda x: str(x).strip() if pd.notna(x) else "")

df = df[
    df[TEXT_COL].ne("") &
    df[MASKED_COL].ne("") &
    df[TARGET_COL].ne("")
].dropna().reset_index(drop=True)

if df.empty:
    raise ValueError("Clean dataset is empty after validation.")

print("✅ Clean dataset loaded")
print("Shape:", df.shape)
display(df.head())

class_distribution = (
    df[TARGET_COL]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

display(class_distribution)

✅ Clean dataset loaded
Shape: (11271, 3)


,body,body_masked,category
0,I often find myself checking on old friends on...,I often find myself checking on old friends on...,Depression
1,this is not a post asking if i have add depres...,this is not a post asking if i have add [CONDI...,ADHD
2,i went to my gp the other day to pick a new pr...,i went to my gp the other day to pick a new pr...,ADHD
3,im a morning person and always envied stories ...,im a morning person and always envied stories ...,ADHD
4,im preparing to be rediagnosed for adhd as im ...,im preparing to be rediagnosed for [CONDITION]...,ADHD


,label,count
0,ADHD,2003
1,Anxiety,1918
2,Bipolar,1824
3,BPD,1618
4,Autism,1497
5,Depression,1433
6,schizophrenia,978


In [20]:
# --- 1.1 Class Distribution ---

class_distribution = class_distribution.sort_values("count", ascending=False).reset_index(drop=True)
total_count = class_distribution["count"].sum()
class_distribution["percent"] = (class_distribution["count"] / total_count * 100).round(2)
class_distribution["label_text"] = (
    class_distribution["count"].astype(str) +
    " (" + class_distribution["percent"].astype(str) + "%)"
)

fig = px.bar(
    class_distribution,
    x="label",
    y="count",
    text="label_text",
    template="plotly_white",
    title="Class Distribution in the Clean Dataset",
    color="count",
    color_continuous_scale="Blues",
)

fig.update_traces(
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Count: %{y}<br>"
        "Percentage: %{customdata[0]}%<extra></extra>"
    ),
    customdata=class_distribution[["percent"]].values,
)

fig.update_layout(
    title=dict(
        text="Class Distribution in the Clean Dataset",
        x=0.5,
        xanchor="center",
        font=dict(size=24)
    ),
    xaxis=dict(
        title="Clinical Category",
        tickangle=-25,
        showgrid=False
    ),
    yaxis=dict(
        title="Number of Samples",
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)"
    ),
    coloraxis_showscale=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    hoverlabel=dict(font_size=13),
    margin=dict(t=90, b=100, l=70, r=40),
    height=550,
)

fig.add_hline(
    y=class_distribution["count"].mean(),
    line_dash="dot",
    line_color="gray",
    annotation_text="Average",
    annotation_position="top right"
)

fig.write_html(str(CLASSICAL_FIGURES_DIR / "clean_dataset_class_distribution.html"))
fig.show()

class_distribution.to_csv(
    CLASSICAL_TABLES_DIR / "clean_dataset_class_distribution.csv",
    index=False
)

print("✅ Clean dataset distribution saved")

✅ Clean dataset distribution saved


## Construction des splits `raw` et `masked`

Le split train/test est construit une seule fois sur les indices, puis appliqué aux deux variantes textuelles afin de garantir une comparaison strictement cohérente.

In [21]:
# ============================================================
# PHASE 2 — TRAIN / TEST SPLIT
# ============================================================

idx = np.arange(len(df))

train_idx, test_idx = train_test_split(
    idx,
    test_size=0.2,
    stratify=df[TARGET_COL],
    random_state=RANDOM_STATE,
)

SPLITS = {
    "raw": {
        "X_train": df.loc[train_idx, TEXT_COL].reset_index(drop=True),
        "X_test": df.loc[test_idx, TEXT_COL].reset_index(drop=True),
        "y_train": df.loc[train_idx, TARGET_COL].reset_index(drop=True),
        "y_test": df.loc[test_idx, TARGET_COL].reset_index(drop=True),
    },
    "masked": {
        "X_train": df.loc[train_idx, MASKED_COL].reset_index(drop=True),
        "X_test": df.loc[test_idx, MASKED_COL].reset_index(drop=True),
        "y_train": df.loc[train_idx, TARGET_COL].reset_index(drop=True),
        "y_test": df.loc[test_idx, TARGET_COL].reset_index(drop=True),
    },
}

print("✅ Train/test split completed")
print("Train size:", len(train_idx))
print("Test size :", len(test_idx))

for variant, split_data in SPLITS.items():
    print(f"\n--- {variant.upper()} ---")
    print(split_data["y_train"].value_counts(normalize=True).round(4))

✅ Train/test split completed
Train size: 9016
Test size : 2255

--- RAW ---
category
ADHD             0.1777
Anxiety          0.1701
Bipolar          0.1618
BPD              0.1435
Autism           0.1329
Depression       0.1271
schizophrenia    0.0868
Name: proportion, dtype: float64

--- MASKED ---
category
ADHD             0.1777
Anxiety          0.1701
Bipolar          0.1618
BPD              0.1435
Autism           0.1329
Depression       0.1271
schizophrenia    0.0868
Name: proportion, dtype: float64


In [22]:
# --- 2.1 Preview of Raw vs Masked Text ---

preview_df = pd.DataFrame({
    "raw_example": SPLITS["raw"]["X_train"].head(5).reset_index(drop=True),
    "masked_example": SPLITS["masked"]["X_train"].head(5).reset_index(drop=True),
})

display(preview_df)

,raw_example,masked_example
0,I have nothing and more importantly nobody lef...,I have nothing and more importantly nobody lef...
1,this is probably more of a neuroscience questi...,this is probably more of a neuroscience questi...
2,Honestly Im not sure what Im looking for with ...,Honestly Im not sure what Im looking for with ...
3,welcome to the 10th edition of win wednesday\n...,welcome to the 10th edition of win wednesday\n...
4,How does psychosis differ in those,How does psychosis differ in those


## Pondération des classes

Les poids de classes sont calculés sur `y_train` afin d’améliorer la sensibilité sur les classes plus difficiles, tout en restant compatibles avec le pipeline existant.

In [23]:
# ============================================================
# PHASE 3 — COMPUTE CUSTOM CLASS WEIGHTS
# ============================================================

from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(SPLITS["masked"]["y_train"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=SPLITS["masked"]["y_train"],
)

class_weight_dict = dict(zip(classes, class_weights))

print("✅ Class weights computed")
print(class_weight_dict)

class_weight_df = pd.DataFrame({
    "label": list(class_weight_dict.keys()),
    "weight": list(class_weight_dict.values()),
}).sort_values("weight", ascending=False)

display(class_weight_df)
class_weight_df.to_csv(CLASSICAL_TABLES_DIR / "class_weight_table.csv", index=False)

✅ Class weights computed
{'ADHD': np.float64(0.8039950062421972), 'Anxiety': np.float64(0.8396349413298566), 'Autism': np.float64(1.0751252086811351), 'BPD': np.float64(0.9953632148377125), 'Bipolar': np.float64(0.8827964359150103), 'Depression': np.float64(1.1239092495636998), 'schizophrenia': np.float64(1.644955300127714)}


,label,weight
6,schizophrenia,1.644955
5,Depression,1.123909
2,Autism,1.075125
3,BPD,0.995363
4,Bipolar,0.882796
1,Anxiety,0.839635
0,ADHD,0.803995


In [24]:
# BOOST HARD CLASSES
if "schizophrenia" in class_weight_dict:
    class_weight_dict["schizophrenia"] *= 1.3

if "Bipolar" in class_weight_dict:
    class_weight_dict["Bipolar"] *= 1.2

print("✅ Boosted class weights")
print(class_weight_dict)

✅ Boosted class weights
{'ADHD': np.float64(0.8039950062421972), 'Anxiety': np.float64(0.8396349413298566), 'Autism': np.float64(1.0751252086811351), 'BPD': np.float64(0.9953632148377125), 'Bipolar': np.float64(1.0593557230980124), 'Depression': np.float64(1.1239092495636998), 'schizophrenia': np.float64(2.1384418901660283)}


In [25]:
# ============================================================
# PHASE 4 — MODEL REGISTRY
# ============================================================

MODEL_REGISTRY = {
    "LinearSVC_balanced": {
        "pipeline": Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
                max_features=50000,
                strip_accents="unicode",
            )),
            ("clf", LinearSVC(
                class_weight=class_weight_dict,
                random_state=RANDOM_STATE,
            )),
        ]),
        "param_grid": {
            "clf__C": [0.5, 1.0, 2.0, 5.0],
        },
    },

    "LinearSVC_plain": {
        "pipeline": Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
                max_features=50000,
                strip_accents="unicode",
            )),
            ("clf", LinearSVC(
                random_state=RANDOM_STATE,
            )),
        ]),
        "param_grid": {
            "clf__C": [0.5, 1.0, 2.0, 5.0],
        },
    },

    "LogReg_balanced": {
        "pipeline": Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
                max_features=50000,
                strip_accents="unicode",
            )),
            ("clf", LogisticRegression(
                solver="saga",
                class_weight=class_weight_dict,
                max_iter=2000,
                random_state=RANDOM_STATE,
            )),
        ]),
        "param_grid": {
            "clf__C": [0.5, 1.0, 2.0, 5.0],
        },
    },

    "LogReg_plain": {
        "pipeline": Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
                max_features=50000,
                strip_accents="unicode",
            )),
            ("clf", LogisticRegression(
                solver="saga",
                max_iter=2000,
                random_state=RANDOM_STATE,
            )),
        ]),
        "param_grid": {
            "clf__C": [0.5, 1.0, 2.0, 5.0],
        },
    },

    "MultinomialNB": {
        "pipeline": Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
                max_features=50000,
                strip_accents="unicode",
            )),
            ("clf", MultinomialNB()),
        ]),
        "param_grid": {
            "clf__alpha": [0.5, 1.0],
        },
    },
}

print("✅ Model registry ready")
print(list(MODEL_REGISTRY.keys()))

✅ Model registry ready
['LinearSVC_balanced', 'LinearSVC_plain', 'LogReg_balanced', 'LogReg_plain', 'MultinomialNB']


In [26]:
# ============================================================
# PHASE 5 — CLINICAL SCORING
# ============================================================

CRITICAL_LABELS = ["Bipolar", "schizophrenia"]

def critical_recall_score(y_true, y_pred, critical_labels=CRITICAL_LABELS):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    recalls = []

    for label in critical_labels:
        if label in report:
            recalls.append(report[label]["recall"])

    if not recalls:
        return 0.0

    return float(np.mean(recalls))

## Validation croisée légère

Cette phase compare rapidement plusieurs modèles classiques sur le jeu d’entraînement, avec une recherche d’hyperparamètres volontairement limitée pour conserver un bon compromis entre robustesse et coût de calcul.

In [27]:
from itertools import product
import random


def sample_param_grid(param_grid, max_candidates=4, random_state=42):
    """
    Sample a limited number of parameter combinations from a sklearn-style grid.

    Args:
        param_grid (dict): Dictionary of hyperparameter lists.
        max_candidates (int): Maximum number of combinations to keep.
        random_state (int): Random seed for reproducibility.

    Returns:
        list[dict]: List of sampled parameter dictionaries.
    """
    if not param_grid:
        return [{}]

    keys = list(param_grid.keys())
    values = [param_grid[key] for key in keys]

    all_candidates = [
        dict(zip(keys, combo))
        for combo in product(*values)
    ]

    if len(all_candidates) <= max_candidates:
        return all_candidates

    rng = random.Random(random_state)
    return rng.sample(all_candidates, max_candidates)

In [28]:
def run_light_cv_benchmark(
    X_train,
    y_train,
    model_registry,
    n_splits=3,
    max_candidates=4,
    random_state=RANDOM_STATE,
):
    """
    Run a lightweight CV benchmark with limited hyperparameter search.
    """
    X_train = pd.Series(X_train).reset_index(drop=True)
    y_train = pd.Series(y_train).reset_index(drop=True)

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    fold_rows = []
    best_params_log = {}

    for model_name, model_spec in model_registry.items():
        print(f"\n🔍 Running model: {model_name}")

        pipeline = model_spec["pipeline"]
        param_grid = model_spec["param_grid"]
        sampled_candidates = sample_param_grid(
            param_grid,
            max_candidates=max_candidates,
            random_state=random_state,
        )

        best_score = -np.inf
        best_params = None

        for params in sampled_candidates:
            fold_f1_scores = []

            for train_idx, valid_idx in cv.split(X_train, y_train):
                X_tr = X_train.iloc[train_idx]
                y_tr = y_train.iloc[train_idx]
                X_val = X_train.iloc[valid_idx]
                y_val = y_train.iloc[valid_idx]

                model = clone(pipeline)
                model.set_params(**params)
                model.fit(X_tr, y_tr)

                y_pred = model.predict(X_val)
                fold_f1 = f1_score(y_val, y_pred, average="macro", zero_division=0)
                fold_f1_scores.append(fold_f1)

            mean_f1 = float(np.mean(fold_f1_scores))

            if mean_f1 > best_score:
                best_score = mean_f1
                best_params = params

        best_params_log[model_name] = best_params

        for fold_id, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train), start=1):
            X_tr = X_train.iloc[train_idx]
            y_tr = y_train.iloc[train_idx]
            X_val = X_train.iloc[valid_idx]
            y_val = y_train.iloc[valid_idx]

            model = clone(pipeline)
            model.set_params(**best_params)
            model.fit(X_tr, y_tr)

            y_pred = model.predict(X_val)

            f1_macro = f1_score(y_val, y_pred, average="macro", zero_division=0)
            recall_macro = recall_score(y_val, y_pred, average="macro", zero_division=0)
            critical_recall = critical_recall_score(y_val, y_pred)

            print(
                f"   Fold {fold_id} | "
                f"F1={f1_macro:.4f} | "
                f"Recall={recall_macro:.4f} | "
                f"Critical={critical_recall:.4f}"
            )

            fold_rows.append({
                "model": model_name,
                "fold": fold_id,
                "f1_macro": f1_macro,
                "recall_macro": recall_macro,
                "critical_recall": critical_recall,
            })

    fold_results = pd.DataFrame(fold_rows)

    summary = (
        fold_results.groupby("model")
        .agg(
            f1_macro_mean=("f1_macro", "mean"),
            recall_macro_mean=("recall_macro", "mean"),
            critical_recall_mean=("critical_recall", "mean"),
            f1_macro_std=("f1_macro", "std"),
            recall_macro_std=("recall_macro", "std"),
            critical_recall_std=("critical_recall", "std"),
        )
        .reset_index()
    )

    for col in [
        "f1_macro_std",
        "recall_macro_std",
        "critical_recall_std",
    ]:
        summary[col] = summary[col].fillna(0)

    summary["robust_score"] = (
        0.4 * summary["f1_macro_mean"]
        + 0.3 * summary["recall_macro_mean"]
        + 0.3 * summary["critical_recall_mean"]
    )

    summary = summary.sort_values(
        by=["robust_score", "critical_recall_mean", "f1_macro_mean"],
        ascending=False,
    ).reset_index(drop=True)

    summary["robust_rank"] = np.arange(1, len(summary) + 1)

    return fold_results, summary, best_params_log

## Exécution du benchmark léger

Le benchmark est exécuté séparément sur les variantes `raw` et `masked`, tout en conservant exactement les mêmes splits et la même logique de scoring.

In [29]:
# ============================================================
# PHASE 7 — RUN LIGHT CV BENCHMARK + SAVE RESULTS
# ============================================================

light_cv_outputs = {}

required_variants = ["raw", "masked"]

for text_variant in required_variants:
    if text_variant not in SPLITS:
        raise KeyError(f"Missing split variant in SPLITS: {text_variant}")

    if "X_train" not in SPLITS[text_variant] or "y_train" not in SPLITS[text_variant]:
        raise KeyError(f"Missing X_train or y_train in SPLITS['{text_variant}']")

    print("\n" + "=" * 80)
    print(f"LIGHT CV BENCHMARK — {text_variant.upper()}")
    print("=" * 80)

    X_train = SPLITS[text_variant]["X_train"]
    y_train = SPLITS[text_variant]["y_train"]

    fold_results, summary, best_params_log = run_light_cv_benchmark(
        X_train=X_train,
        y_train=y_train,
        model_registry=MODEL_REGISTRY,
        n_splits=3,
        max_candidates=4,
        random_state=RANDOM_STATE,
    )

    summary["protocol"] = "light_cv"
    summary["text_variant"] = text_variant

    light_cv_outputs[text_variant] = {
        "fold_results": fold_results,
        "summary": summary,
        "best_params_log": best_params_log,
    }

light_cv_summary = pd.concat(
    [
        light_cv_outputs["raw"]["summary"],
        light_cv_outputs["masked"]["summary"],
    ],
    axis=0,
    ignore_index=True,
)

light_cv_summary = light_cv_summary.sort_values(
    by=["robust_score", "critical_recall_mean", "f1_macro_mean"],
    ascending=[False, False, False],
).reset_index(drop=True)

display(light_cv_summary)

light_cv_summary.to_csv(NORMAL_CV_RESULTS_PATH, index=False)


print("✅ Light CV benchmark completed and saved")
print("NORMAL_CV_RESULTS_PATH:", NORMAL_CV_RESULTS_PATH)



LIGHT CV BENCHMARK — RAW

🔍 Running model: LinearSVC_balanced
   Fold 1 | F1=0.7738 | Recall=0.7732 | Critical=0.7055
   Fold 2 | F1=0.7619 | Recall=0.7604 | Critical=0.6743
   Fold 3 | F1=0.7699 | Recall=0.7674 | Critical=0.6734

🔍 Running model: LinearSVC_plain
   Fold 1 | F1=0.7776 | Recall=0.7728 | Critical=0.6835
   Fold 2 | F1=0.7574 | Recall=0.7524 | Critical=0.6302
   Fold 3 | F1=0.7652 | Recall=0.7594 | Critical=0.6401

🔍 Running model: LogReg_balanced
   Fold 1 | F1=0.7613 | Recall=0.7616 | Critical=0.6974
   Fold 2 | F1=0.7478 | Recall=0.7472 | Critical=0.6628
   Fold 3 | F1=0.7582 | Recall=0.7573 | Critical=0.6740

🔍 Running model: LogReg_plain
   Fold 1 | F1=0.7655 | Recall=0.7605 | Critical=0.6655
   Fold 2 | F1=0.7388 | Recall=0.7341 | Critical=0.5966
   Fold 3 | F1=0.7576 | Recall=0.7521 | Critical=0.6280

🔍 Running model: MultinomialNB
   Fold 1 | F1=0.5608 | Recall=0.5790 | Critical=0.3527
   Fold 2 | F1=0.5321 | Recall=0.5523 | Critical=0.3382
   Fold 3 | F1=0.5430 

,model,f1_macro_mean,recall_macro_mean,critical_recall_mean,f1_macro_std,recall_macro_std,critical_recall_std,robust_score,robust_rank,protocol,text_variant
0,LinearSVC_balanced,0.768516,0.767001,0.684379,0.006075,0.006391,0.018273,0.742820,1,light_cv,raw
1,LogReg_balanced,0.755790,0.755371,0.678057,0.007061,0.007396,0.017656,0.732344,2,light_cv,raw
2,LinearSVC_plain,0.766738,0.761514,0.651268,0.010210,0.010377,0.028352,0.730530,3,light_cv,raw
3,LogReg_plain,0.753981,0.748879,0.630051,0.013695,0.013482,0.034486,0.715271,4,light_cv,raw
4,LinearSVC_balanced,0.692175,0.691903,0.597883,0.020567,0.020627,0.027198,0.663806,1,light_cv,masked
5,LogReg_balanced,0.683549,0.683117,0.593510,0.018286,0.018493,0.024709,0.656408,2,light_cv,masked
6,LinearSVC_plain,0.685336,0.681800,0.564224,0.017256,0.016945,0.032583,0.647942,3,light_cv,masked
7,LogReg_plain,0.674757,0.670416,0.530150,0.022708,0.022341,0.036157,0.630072,4,light_cv,masked
8,MultinomialNB,0.545324,0.566147,0.350395,0.014490,0.013413,0.011194,0.493092,5,light_cv,raw
9,MultinomialNB,0.492918,0.516509,0.311855,0.013158,0.011025,0.014266,0.445677,5,light_cv,masked


✅ Light CV benchmark completed and saved
NORMAL_CV_RESULTS_PATH: C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\normal_cv_summary.csv


In [30]:
# --- Visual comparison of light CV results ---

plot_df = light_cv_summary.copy()

fig = px.bar(
    plot_df,
    x="model",
    y="robust_score",
    color="text_variant",
    barmode="group",
    template=PLOTLY_TEMPLATE,
    title="Comparaison des modèles — Light CV",
    hover_data=[
        "f1_macro_mean",
        "recall_macro_mean",
        "critical_recall_mean",
        "robust_rank",
    ],
)

fig.update_layout(
    xaxis_title="Modèle",
    yaxis_title="Robust score",
    title_x=0.5,
)

fig.write_html(str(CLASSICAL_FIGURES_DIR / "light_cv_comparison.html"))
fig.show()

print("✅ Light CV comparison figure saved")

✅ Light CV comparison figure saved


## Nested CV léger

Cette étape ajoute une évaluation plus rigoureuse sur le jeu d’entraînement, avec une boucle interne pour la sélection d’hyperparamètres et une boucle externe pour estimer la robustesse réelle.

In [31]:
def run_light_nested_cv_benchmark(
    X_train,
    y_train,
    model_registry,
    outer_splits=3,
    inner_splits=2,
    max_candidates=3,
    random_state=RANDOM_STATE,
):
    """
    Run a lightweight nested CV benchmark.
    Inner-loop hyperparameter selection is aligned with the clinical robust score.
    """
    X_train = pd.Series(X_train).reset_index(drop=True)
    y_train = pd.Series(y_train).reset_index(drop=True)

    outer_cv = StratifiedKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=random_state,
    )

    nested_rows = []
    nested_best_params = {}

    for model_name, model_spec in model_registry.items():
        print(f"\n🧪 Nested CV model: {model_name}")

        pipeline = model_spec["pipeline"]
        param_grid = model_spec["param_grid"]

        nested_best_params[model_name] = []

        for outer_fold, (dev_idx, test_idx) in enumerate(outer_cv.split(X_train, y_train), start=1):
            X_dev = X_train.iloc[dev_idx]
            y_dev = y_train.iloc[dev_idx]
            X_outer_test = X_train.iloc[test_idx]
            y_outer_test = y_train.iloc[test_idx]

            inner_cv = StratifiedKFold(
                n_splits=inner_splits,
                shuffle=True,
                random_state=random_state,
            )

            sampled_candidates = sample_param_grid(
                param_grid,
                max_candidates=max_candidates,
                random_state=random_state,
            )

            best_score = -np.inf
            best_params = None

            for params in sampled_candidates:
                inner_scores = []

                for inner_train_idx, inner_valid_idx in inner_cv.split(X_dev, y_dev):
                    X_inner_train = X_dev.iloc[inner_train_idx]
                    y_inner_train = y_dev.iloc[inner_train_idx]
                    X_inner_valid = X_dev.iloc[inner_valid_idx]
                    y_inner_valid = y_dev.iloc[inner_valid_idx]

                    model = clone(pipeline)
                    model.set_params(**params)
                    model.fit(X_inner_train, y_inner_train)

                    y_pred_inner = model.predict(X_inner_valid)

                    f1_macro = f1_score(y_inner_valid, y_pred_inner, average="macro", zero_division=0)
                    recall_macro = recall_score(y_inner_valid, y_pred_inner, average="macro", zero_division=0)
                    critical_recall = critical_recall_score(y_inner_valid, y_pred_inner)

                    robust_score = (
                        0.4 * critical_recall
                        + 0.3 * recall_macro
                        + 0.3 * f1_macro
                    )
                    inner_scores.append(robust_score)

                mean_inner_score = float(np.mean(inner_scores))

                if mean_inner_score > best_score:
                    best_score = mean_inner_score
                    best_params = params

            nested_best_params[model_name].append({
                "outer_fold": outer_fold,
                "best_params": best_params,
            })

            final_model = clone(pipeline)
            final_model.set_params(**best_params)
            final_model.fit(X_dev, y_dev)

            y_outer_pred = final_model.predict(X_outer_test)

            f1_macro = f1_score(y_outer_test, y_outer_pred, average="macro", zero_division=0)
            recall_macro = recall_score(y_outer_test, y_outer_pred, average="macro", zero_division=0)
            critical_recall = critical_recall_score(y_outer_test, y_outer_pred)

            print(
                f"   Outer fold {outer_fold} | "
                f"F1={f1_macro:.4f} | "
                f"Recall={recall_macro:.4f} | "
                f"Critical={critical_recall:.4f}"
            )

            nested_rows.append({
                "model": model_name,
                "outer_fold": outer_fold,
                "f1_macro": f1_macro,
                "recall_macro": recall_macro,
                "critical_recall": critical_recall,
            })

    nested_fold_results = pd.DataFrame(nested_rows)

    nested_summary = (
        nested_fold_results.groupby("model")
        .agg(
            f1_macro_mean=("f1_macro", "mean"),
            recall_macro_mean=("recall_macro", "mean"),
            critical_recall_mean=("critical_recall", "mean"),
            f1_macro_std=("f1_macro", "std"),
            recall_macro_std=("recall_macro", "std"),
            critical_recall_std=("critical_recall", "std"),
        )
        .reset_index()
    )

    for col in ["f1_macro_std", "recall_macro_std", "critical_recall_std"]:
        nested_summary[col] = nested_summary[col].fillna(0)

    nested_summary["robust_score"] = (
        0.4 * nested_summary["critical_recall_mean"]
        + 0.3 * nested_summary["recall_macro_mean"]
        + 0.3 * nested_summary["f1_macro_mean"]
    )

    nested_summary = nested_summary.sort_values(
        by=["robust_score", "critical_recall_mean", "f1_macro_mean"],
        ascending=False,
    ).reset_index(drop=True)

    nested_summary["robust_rank"] = np.arange(1, len(nested_summary) + 1)

    return nested_fold_results, nested_summary, nested_best_params

In [32]:
# ============================================================
# PHASE 8 — RUN LIGHT NESTED CV BENCHMARK
# ============================================================

nested_cv_outputs = {}

for text_variant in ["raw", "masked"]:
    print("\n" + "=" * 80)
    print(f"LIGHT NESTED CV — {text_variant.upper()}")
    print("=" * 80)

    X_train = SPLITS[text_variant]["X_train"]
    y_train = SPLITS[text_variant]["y_train"]

    nested_fold_results, nested_summary, nested_best_params = run_light_nested_cv_benchmark(
        X_train=X_train,
        y_train=y_train,
        model_registry=MODEL_REGISTRY,
        outer_splits=3,
        inner_splits=2,
        max_candidates=3,
        random_state=RANDOM_STATE,
    )

    nested_summary["protocol"] = "nested_cv"
    nested_summary["text_variant"] = text_variant

    nested_cv_outputs[text_variant] = {
        "fold_results": nested_fold_results,
        "summary": nested_summary,
        "best_params": nested_best_params,
    }

nested_cv_summary = pd.concat(
    [
        nested_cv_outputs["raw"]["summary"],
        nested_cv_outputs["masked"]["summary"],
    ],
    axis=0,
    ignore_index=True,
)

nested_cv_summary = nested_cv_summary.sort_values(
    by=["robust_score", "critical_recall_mean", "f1_macro_mean"],
    ascending=[False, False, False],
).reset_index(drop=True)

display(nested_cv_summary)

nested_cv_summary.to_csv(NESTED_CV_RESULTS_PATH, index=False)
nested_cv_summary.to_csv(CLASSICAL_TABLES_DIR / "nested_cv_summary.csv", index=False)

print("✅ Nested CV benchmark completed and saved")
print("NESTED_CV_RESULTS_PATH:", NESTED_CV_RESULTS_PATH)
print("TABLE COPY:", CLASSICAL_TABLES_DIR / "nested_cv_summary.csv")


LIGHT NESTED CV — RAW

🧪 Nested CV model: LinearSVC_balanced
   Outer fold 1 | F1=0.7738 | Recall=0.7732 | Critical=0.7055
   Outer fold 2 | F1=0.7619 | Recall=0.7604 | Critical=0.6743
   Outer fold 3 | F1=0.7699 | Recall=0.7674 | Critical=0.6734

🧪 Nested CV model: LinearSVC_plain
   Outer fold 1 | F1=0.7656 | Recall=0.7617 | Critical=0.6888
   Outer fold 2 | F1=0.7494 | Recall=0.7461 | Critical=0.6397
   Outer fold 3 | F1=0.7586 | Recall=0.7540 | Critical=0.6486

🧪 Nested CV model: LogReg_balanced
   Outer fold 1 | F1=0.7204 | Recall=0.7260 | Critical=0.7178
   Outer fold 2 | F1=0.7271 | Recall=0.7310 | Critical=0.6942
   Outer fold 3 | F1=0.7271 | Recall=0.7324 | Critical=0.7147

🧪 Nested CV model: LogReg_plain
   Outer fold 1 | F1=0.7655 | Recall=0.7605 | Critical=0.6655
   Outer fold 2 | F1=0.7388 | Recall=0.7341 | Critical=0.5966
   Outer fold 3 | F1=0.7576 | Recall=0.7521 | Critical=0.6280

🧪 Nested CV model: MultinomialNB
   Outer fold 1 | F1=0.5608 | Recall=0.5790 | Critical=

,model,f1_macro_mean,recall_macro_mean,critical_recall_mean,f1_macro_std,recall_macro_std,critical_recall_std,robust_score,robust_rank,protocol,text_variant
0,LinearSVC_balanced,0.768516,0.767001,0.684379,0.006075,0.006391,0.018273,0.734407,1,nested_cv,raw
1,LogReg_balanced,0.724886,0.729826,0.708881,0.003861,0.003367,0.012813,0.719966,2,nested_cv,raw
2,LinearSVC_plain,0.757862,0.753905,0.659020,0.008144,0.007790,0.026159,0.717138,3,nested_cv,raw
3,LogReg_plain,0.753981,0.748879,0.630051,0.013695,0.013482,0.034486,0.702878,4,nested_cv,raw
4,LinearSVC_balanced,0.692175,0.691903,0.597883,0.020567,0.020627,0.027198,0.654376,1,nested_cv,masked
5,LogReg_balanced,0.634042,0.641166,0.631647,0.013316,0.013136,0.017820,0.635221,2,nested_cv,masked
6,LinearSVC_plain,0.674886,0.671780,0.568944,0.014612,0.014416,0.025769,0.631577,3,nested_cv,masked
7,LogReg_plain,0.674757,0.670416,0.530150,0.022708,0.022341,0.036157,0.615612,4,nested_cv,masked
8,MultinomialNB,0.545324,0.566147,0.350395,0.014490,0.013413,0.011194,0.473599,5,nested_cv,raw
9,MultinomialNB,0.492918,0.516509,0.311855,0.013158,0.011025,0.014266,0.427570,5,nested_cv,masked


✅ Nested CV benchmark completed and saved
NESTED_CV_RESULTS_PATH: C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\nested_cv_summary.csv
TABLE COPY: C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\nested_cv_summary.csv


In [33]:
# ============================================================
# PHASE 9 — GLOBAL COMPARISON TABLE
# ============================================================

comparison_df = pd.concat(
    [light_cv_summary, nested_cv_summary],
    axis=0,
    ignore_index=True,
)

comparison_df = comparison_df.sort_values(
    by=["robust_score", "critical_recall_mean", "f1_macro_mean"],
    ascending=[False, False, False],
).reset_index(drop=True)

comparison_df["global_rank"] = np.arange(1, len(comparison_df) + 1)

display(comparison_df)

comparison_df.to_csv(COMPARISON_RESULTS_PATH, index=False)
comparison_df.to_csv(CLASSICAL_TABLES_DIR / "classical_model_comparison.csv", index=False)

print("✅ Global comparison table saved")
print("COMPARISON_RESULTS_PATH:", COMPARISON_RESULTS_PATH)
print("TABLE COPY:", CLASSICAL_TABLES_DIR / "classical_model_comparison.csv")

,model,f1_macro_mean,recall_macro_mean,critical_recall_mean,f1_macro_std,recall_macro_std,critical_recall_std,robust_score,robust_rank,protocol,text_variant,global_rank
0,LinearSVC_balanced,0.768516,0.767001,0.684379,0.006075,0.006391,0.018273,0.742820,1,light_cv,raw,1
1,LinearSVC_balanced,0.768516,0.767001,0.684379,0.006075,0.006391,0.018273,0.734407,1,nested_cv,raw,2
2,LogReg_balanced,0.755790,0.755371,0.678057,0.007061,0.007396,0.017656,0.732344,2,light_cv,raw,3
3,LinearSVC_plain,0.766738,0.761514,0.651268,0.010210,0.010377,0.028352,0.730530,3,light_cv,raw,4
4,LogReg_balanced,0.724886,0.729826,0.708881,0.003861,0.003367,0.012813,0.719966,2,nested_cv,raw,5
5,LinearSVC_plain,0.757862,0.753905,0.659020,0.008144,0.007790,0.026159,0.717138,3,nested_cv,raw,6
6,LogReg_plain,0.753981,0.748879,0.630051,0.013695,0.013482,0.034486,0.715271,4,light_cv,raw,7
7,LogReg_plain,0.753981,0.748879,0.630051,0.013695,0.013482,0.034486,0.702878,4,nested_cv,raw,8
8,LinearSVC_balanced,0.692175,0.691903,0.597883,0.020567,0.020627,0.027198,0.663806,1,light_cv,masked,9
9,LogReg_balanced,0.683549,0.683117,0.593510,0.018286,0.018493,0.024709,0.656408,2,light_cv,masked,10


✅ Global comparison table saved
COMPARISON_RESULTS_PATH: C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\classical_model_comparison.csv
TABLE COPY: C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\classical_model_comparison.csv


## Comparaison consolidée

Les résultats issus de `light_cv` et `nested_cv` sont regroupés ici afin de soutenir une sélection de modèle plus sérieuse et plus lisible.

In [34]:
fig = px.bar(
    comparison_df,
    x="model",
    y="robust_score",
    color="text_variant",
    facet_col="protocol",
    barmode="group",
    template=PLOTLY_TEMPLATE,
    title="Comparaison globale des modèles classiques",
    hover_data=[
        "f1_macro_mean",
        "recall_macro_mean",
        "critical_recall_mean",
        "robust_rank",
    ],
)

fig.update_layout(
    xaxis_title="Modèle",
    yaxis_title="Robust score",
    title_x=0.5,
)

fig.write_html(str(CLASSICAL_FIGURES_DIR / "global_classical_comparison.html"))
fig.show()

print("✅ Global comparison figure saved")

✅ Global comparison figure saved


## Sélection du modèle champion

Le choix final repose en priorité sur les résultats `nested_cv`, avec une attention particulière portée à :
- `critical_recall_mean`
- `f1_macro_mean`
- stabilité globale (`robust_score`)

In [35]:
# ============================================================
# PHASE 10 — CHAMPION MODEL SELECTION
# ============================================================

champion_candidates = nested_cv_summary.copy()

champion_candidates = champion_candidates.sort_values(
    by=["robust_score", "critical_recall_mean", "f1_macro_mean"],
    ascending=False,
).reset_index(drop=True)

display(champion_candidates)

CHAMPION_ROW = champion_candidates.iloc[0].to_dict()
CHAMPION_MODEL_NAME = CHAMPION_ROW["model"]
CHAMPION_TEXT_VARIANT = CHAMPION_ROW["text_variant"]

print("✅ Champion selected")
print("CHAMPION_MODEL_NAME :", CHAMPION_MODEL_NAME)
print("CHAMPION_TEXT_VARIANT:", CHAMPION_TEXT_VARIANT)

,model,f1_macro_mean,recall_macro_mean,critical_recall_mean,f1_macro_std,recall_macro_std,critical_recall_std,robust_score,robust_rank,protocol,text_variant
0,LinearSVC_balanced,0.768516,0.767001,0.684379,0.006075,0.006391,0.018273,0.734407,1,nested_cv,raw
1,LogReg_balanced,0.724886,0.729826,0.708881,0.003861,0.003367,0.012813,0.719966,2,nested_cv,raw
2,LinearSVC_plain,0.757862,0.753905,0.659020,0.008144,0.007790,0.026159,0.717138,3,nested_cv,raw
3,LogReg_plain,0.753981,0.748879,0.630051,0.013695,0.013482,0.034486,0.702878,4,nested_cv,raw
4,LinearSVC_balanced,0.692175,0.691903,0.597883,0.020567,0.020627,0.027198,0.654376,1,nested_cv,masked
5,LogReg_balanced,0.634042,0.641166,0.631647,0.013316,0.013136,0.017820,0.635221,2,nested_cv,masked
6,LinearSVC_plain,0.674886,0.671780,0.568944,0.014612,0.014416,0.025769,0.631577,3,nested_cv,masked
7,LogReg_plain,0.674757,0.670416,0.530150,0.022708,0.022341,0.036157,0.615612,4,nested_cv,masked
8,MultinomialNB,0.545324,0.566147,0.350395,0.014490,0.013413,0.011194,0.473599,5,nested_cv,raw
9,MultinomialNB,0.492918,0.516509,0.311855,0.013158,0.011025,0.014266,0.427570,5,nested_cv,masked


✅ Champion selected
CHAMPION_MODEL_NAME : LinearSVC_balanced
CHAMPION_TEXT_VARIANT: raw


In [36]:
from collections import Counter
import json

def _make_params_hashable(params):
    return json.dumps(params, sort_keys=True)

def select_nested_champion_params(nested_cv_outputs, text_variant, model_name):
    params_list = nested_cv_outputs[text_variant]["best_params"][model_name]
    extracted_params = [item["best_params"] for item in params_list if item["best_params"] is not None]

    if not extracted_params:
        raise ValueError(f"No nested params found for {model_name} / {text_variant}")

    param_counter = Counter(_make_params_hashable(p) for p in extracted_params)
    most_common_params_str = param_counter.most_common(1)[0][0]
    return json.loads(most_common_params_str)

CHAMPION_PARAMS = select_nested_champion_params(
    nested_cv_outputs=nested_cv_outputs,
    text_variant=CHAMPION_TEXT_VARIANT,
    model_name=CHAMPION_MODEL_NAME,
)

print("✅ Champion params loaded from nested CV")
print(CHAMPION_PARAMS)

✅ Champion params loaded from nested CV
{'clf__C': 0.5}


## Conclusion intermédiaire

Le modèle champion retenu à l’issue du `nested_cv` correspond au meilleur compromis entre :
- performance macro,
- rappel global,
- rappel sur les classes critiques,
- stabilité inter-folds.

Ce modèle sera retenu comme **champion principal** pour l’évaluation finale sur le jeu de test figé.

La variante non retenue reste utile comme **référence méthodologique**, afin de documenter l’impact du masquage lexical sur la robustesse et le risque de fuite d’information.

# ----------------------------------
## Entraînement final et évaluation sur le jeu de test
# -----------------------------------
Cette dernière phase entraîne le modèle champion sur l’ensemble du jeu d’entraînement, puis l’évalue une seule fois sur le jeu de test figé.

In [37]:
# ============================================================
# PHASE 11 — FINAL TRAIN / TEST DATA FOR CHAMPION
# ============================================================

X_train_final = SPLITS[CHAMPION_TEXT_VARIANT]["X_train"].reset_index(drop=True)
X_test_final = SPLITS[CHAMPION_TEXT_VARIANT]["X_test"].reset_index(drop=True)
y_train_final = SPLITS[CHAMPION_TEXT_VARIANT]["y_train"].reset_index(drop=True)
y_test_final = SPLITS[CHAMPION_TEXT_VARIANT]["y_test"].reset_index(drop=True)

print("✅ Final train/test data prepared")
print("CHAMPION_MODEL_NAME :", CHAMPION_MODEL_NAME)
print("CHAMPION_TEXT_VARIANT:", CHAMPION_TEXT_VARIANT)
print("Train size:", len(X_train_final))
print("Test size :", len(X_test_final))

✅ Final train/test data prepared
CHAMPION_MODEL_NAME : LinearSVC_balanced
CHAMPION_TEXT_VARIANT: raw
Train size: 9016
Test size : 2255


In [38]:
# ============================================================
# PHASE 12 — FINAL MODEL TRAINING
# ============================================================

final_model = clone(MODEL_REGISTRY[CHAMPION_MODEL_NAME]["pipeline"])
final_model.set_params(**CHAMPION_PARAMS)
final_model.fit(X_train_final, y_train_final)

print("✅ Final model trained successfully")
print(final_model)

✅ Final model trained successfully
Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, max_features=50000, min_df=2,
                                 ngram_range=(1, 2), strip_accents='unicode',
                                 sublinear_tf=True)),
                ('clf',
                 LinearSVC(C=0.5,
                           class_weight={'ADHD': np.float64(0.8039950062421972),
                                         'Anxiety': np.float64(0.8396349413298566),
                                         'Autism': np.float64(1.0751252086811351),
                                         'BPD': np.float64(0.9953632148377125),
                                         'Bipolar': np.float64(1.0593557230980124),
                                         'Depression': np.float64(1.1239092495636998),
                                         'schizophrenia': np.float64(2.1384418901660283)},
                           random_state=42))])


In [39]:
# ============================================================
# PHASE 13 — FINAL TEST PREDICTIONS
# ============================================================

y_test_pred = final_model.predict(X_test_final)

print("✅ Test predictions generated")
print("Prediction count:", len(y_test_pred))

✅ Test predictions generated
Prediction count: 2255


In [40]:
# ============================================================
# PHASE 14 — FINAL TEST METRICS
# ============================================================

final_f1_macro = f1_score(
    y_test_final,
    y_test_pred,
    average="macro",
    zero_division=0,
)

final_recall_macro = recall_score(
    y_test_final,
    y_test_pred,
    average="macro",
    zero_division=0,
)

final_critical_recall = critical_recall_score(
    y_test_final,
    y_test_pred,
)

final_test_metrics = pd.DataFrame([{
    "champion_model": CHAMPION_MODEL_NAME,
    "text_variant": CHAMPION_TEXT_VARIANT,
    "f1_macro": round(final_f1_macro, 6),
    "recall_macro": round(final_recall_macro, 6),
    "critical_recall": round(final_critical_recall, 6),
}])

display(final_test_metrics)

final_test_metrics.to_csv(FINAL_TEST_RESULTS_PATH, index=False)
final_test_metrics.to_csv(CLASSICAL_TABLES_DIR / "final_test_metrics.csv", index=False)

print("✅ Final test metrics saved")
print(FINAL_TEST_RESULTS_PATH)

,champion_model,text_variant,f1_macro,recall_macro,critical_recall
0,LinearSVC_balanced,raw,0.778927,0.778686,0.698068


✅ Final test metrics saved
C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\final_test_metrics.csv


In [41]:
# ============================================================
# PHASE 15 — CLASSIFICATION REPORT
# ============================================================

report_dict = classification_report(
    y_test_final,
    y_test_pred,
    output_dict=True,
    zero_division=0,
)

classification_report_df = pd.DataFrame(report_dict).transpose().reset_index()
classification_report_df = classification_report_df.rename(columns={"index": "label"})

display(classification_report_df)

classification_report_df.to_csv(
    CLASSICAL_TABLES_DIR / "classification_report_final_test.csv",
    index=False,
)

print("✅ Classification report saved")

,label,precision,recall,f1-score,support
0,ADHD,0.877108,0.907731,0.892157,401.0000
1,Anxiety,0.808108,0.778646,0.793103,384.0000
2,Autism,0.843333,0.846154,0.844741,299.0000
3,BPD,0.817337,0.814815,0.816074,324.0000
4,Bipolar,0.810811,0.739726,0.773639,365.0000
5,Depression,0.617021,0.707317,0.659091,287.0000
6,schizophrenia,0.691892,0.656410,0.673684,195.0000
7,accuracy,0.789800,0.789800,0.789800,0.7898
8,macro avg,0.780802,0.778686,0.778927,2255.0000
9,weighted avg,0.792443,0.789800,0.790332,2255.0000


✅ Classification report saved


In [42]:
# ============================================================
# PHASE 16 — CLASS RECALL ANALYSIS
# ============================================================

class_recall_df = classification_report_df.copy()

class_recall_df = class_recall_df[
    class_recall_df["label"].isin(sorted(y_test_final.unique()))
].copy()

class_recall_df = class_recall_df[["label", "precision", "recall", "f1-score", "support"]]
class_recall_df = class_recall_df.sort_values("recall", ascending=False).reset_index(drop=True)

display(class_recall_df)

class_recall_df.to_csv(CLASS_RECALL_RESULTS_PATH, index=False)
class_recall_df.to_csv(CLASSICAL_TABLES_DIR / "class_recall_results.csv", index=False)

print("✅ Class recall results saved")
print(CLASS_RECALL_RESULTS_PATH)

,label,precision,recall,f1-score,support
0,ADHD,0.877108,0.907731,0.892157,401.0
1,Autism,0.843333,0.846154,0.844741,299.0
2,BPD,0.817337,0.814815,0.816074,324.0
3,Anxiety,0.808108,0.778646,0.793103,384.0
4,Bipolar,0.810811,0.739726,0.773639,365.0
5,Depression,0.617021,0.707317,0.659091,287.0
6,schizophrenia,0.691892,0.656410,0.673684,195.0


✅ Class recall results saved
C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\class_recall_results.csv


In [43]:
fig = px.bar(
    class_recall_df,
    x="label",
    y="recall",
    text="recall",
    template=PLOTLY_TEMPLATE,
    title="Recall par classe — modèle champion",
    hover_data=["precision", "f1-score", "support"],
)

fig.update_layout(
    xaxis_title="Classe",
    yaxis_title="Recall",
    title_x=0.5,
)

fig.write_html(str(CLASSICAL_FIGURES_DIR / "class_recall_final_test.html"))
fig.show()

print("✅ Class recall figure saved")

✅ Class recall figure saved


In [44]:
# ============================================================
# PHASE 17 — CONFUSION MATRIX
# ============================================================

labels = sorted(y_test_final.unique())

cm = confusion_matrix(
    y_test_final,
    y_test_pred,
    labels=labels,
)

cm_df = pd.DataFrame(cm, index=labels, columns=labels)

display(cm_df)

cm_df.to_csv(CLASSICAL_TABLES_DIR / "confusion_matrix_final_test.csv")

fig = px.imshow(
    cm_df,
    text_auto=True,
    aspect="auto",
    template=PLOTLY_TEMPLATE,
    title="Matrice de confusion — modèle champion",
)

fig.update_layout(
    xaxis_title="Predicted label",
    yaxis_title="True label",
    title_x=0.5,
)

fig.write_html(str(CLASSICAL_FIGURES_DIR / "confusion_matrix_final_test.html"))
fig.show()

print("✅ Confusion matrix saved")

,ADHD,Anxiety,Autism,BPD,Bipolar,Depression,schizophrenia
ADHD,364,7,10,1,3,12,4
Anxiety,11,299,12,8,6,35,13
Autism,11,9,253,6,8,8,4
BPD,6,9,4,264,17,20,4
Bipolar,7,12,6,22,270,34,14
Depression,9,23,8,15,11,203,18
schizophrenia,7,11,7,7,18,17,128


✅ Confusion matrix saved


In [45]:
# ============================================================
# PHASE 18 — EXPORT FINAL PREDICTIONS
# ============================================================

raw_test_text = SPLITS["raw"]["X_test"].reset_index(drop=True)

final_predictions_df = pd.DataFrame({
    "text": raw_test_text,
    "text_used_for_model": X_test_final.reset_index(drop=True),
    "y_true": y_test_final.reset_index(drop=True),
    "y_pred": pd.Series(y_test_pred).reset_index(drop=True),
})

final_predictions_df["is_correct"] = (
    final_predictions_df["y_true"] == final_predictions_df["y_pred"]
)

final_predictions_df.to_csv(FINAL_PREDICTIONS_PATH, index=False)
final_predictions_df.to_csv(CLASSICAL_TABLES_DIR / "final_test_predictions.csv", index=False)

display(final_predictions_df.head(10))

print("✅ Final predictions exported")
print(FINAL_PREDICTIONS_PATH)

,text,text_used_for_model,y_true,y_pred,is_correct
0,My first day of JR year starts tomorrow and I ...,My first day of JR year starts tomorrow and I ...,Anxiety,Anxiety,True
1,Its one of those days where I feel like garbag...,Its one of those days where I feel like garbag...,Anxiety,Anxiety,True
2,a while back i stumbled across a ted talk abou...,a while back i stumbled across a ted talk abou...,ADHD,ADHD,True
3,I was having cardiac anxiety amp after 1 year ...,I was having cardiac anxiety amp after 1 year ...,Anxiety,Anxiety,True
4,My anxiety has tormented me for the most of my...,My anxiety has tormented me for the most of my...,Anxiety,Anxiety,True
5,Im 43 and have cried in front of EVERY boss Iv...,Im 43 and have cried in front of EVERY boss Iv...,Depression,Depression,True
6,My husband was diagnosed bipolar 2 a year ago ...,My husband was diagnosed bipolar 2 a year ago ...,Bipolar,Bipolar,True
7,Does anyone else feel like when they go throug...,Does anyone else feel like when they go throug...,Bipolar,Bipolar,True
8,I feel so stupid but I just go yes yes yes yes...,I feel so stupid but I just go yes yes yes yes...,Bipolar,Autism,False
9,Sorry I dont know how to explain this properly...,Sorry I dont know how to explain this properly...,Depression,Depression,True


✅ Final predictions exported
C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\final_test_predictions.csv


In [46]:
# ============================================================
# PHASE 19 — SAVE FINAL MODEL
# ============================================================

joblib.dump(final_model, FINAL_MODEL_PATH)

print("✅ Final model saved")
print(FINAL_MODEL_PATH)

✅ Final model saved
C:\Users\anafi\Desktop\final_project_jedha\models\best_classical_model.joblib


In [47]:
# ============================================================
# PHASE 20 — FINAL SUMMARY
# ============================================================

notebook_2_summary = pd.DataFrame({
    "item": [
        "Champion model",
        "Text variant",
        "Final test f1_macro",
        "Final test recall_macro",
        "Final test critical_recall",
        "Saved model path",
        "Predictions path",
    ],
    "value": [
        CHAMPION_MODEL_NAME,
        CHAMPION_TEXT_VARIANT,
        round(final_f1_macro, 6),
        round(final_recall_macro, 6),
        round(final_critical_recall, 6),
        str(FINAL_MODEL_PATH),
        str(FINAL_PREDICTIONS_PATH),
    ],
})

display(notebook_2_summary)

notebook_2_summary.to_csv(
    CLASSICAL_TABLES_DIR / "notebook_2_final_summary.csv",
    index=False,
)

print("✅ Notebook 2 completed successfully")

,item,value
0,Champion model,LinearSVC_balanced
1,Text variant,raw
2,Final test f1_macro,0.778927
3,Final test recall_macro,0.778686
4,Final test critical_recall,0.698068
5,Saved model path,C:\Users\anafi\Desktop\final_project_jedha\mod...
6,Predictions path,C:\Users\anafi\Desktop\final_project_jedha\rep...


✅ Notebook 2 completed successfully


## Conclusion du benchmark classique

Le modèle champion retenu à l’issue de ce notebook est le meilleur compromis entre :
- performance macro,
- rappel global,
- rappel sur les classes critiques,
- stabilité inter-folds.

Les exports générés ici seront réutilisés dans le notebook d’évaluation clinique et d’analyse d’erreurs.